In [1]:
import os.path

import pandas as pd
import numpy as np
import xarray as xr


INPUT_DIR = "/run/media/martin/KINGSTON/NDVI/test_area"
CSV_INVENTORY_PATH = "/run/media/martin/KINGSTON/NDVI/test_area/log.csv"

OUT_COMPOSITES_DIR = "/run/media/martin/KINGSTON/NDVI/test_area_composites"
LOG_PATH = os.path.join(OUT_COMPOSITES_DIR, "log.csv")

OUT_NO_NAN_DIR = "/run/media/martin/KINGSTON/NDVI/test_area_no_nan"
LOG_PATH_NO_NAN = os.path.join(OUT_NO_NAN_DIR, "log.csv")

In [2]:
inventory = pd.read_csv(CSV_INVENTORY_PATH, index_col=0, header = 0)

min_year = inventory.year.min()
max_year = inventory.year.max()

min_month = inventory.month.min()
max_month = inventory.month.max() 

#### Create Monthly Composites - Matrix [year, month]

In [5]:
inventory = pd.read_csv(CSV_INVENTORY_PATH, index_col=0, header = 0)

min_year = inventory.year.min()
max_year = inventory.year.max()

min_month = inventory.month.min()
max_month = inventory.month.max() # assumption - all year have observations from same moths

out_db = pd.DataFrame(columns=[ "year", "month", "path"])


for year in range(min_year,max_year+1):
    year_observations = inventory[inventory["year"] == year] # all months in one year
    #print("year_observations\n",year_observations)
    
    for month in range(min_month,max_month+1):
        month_observations = year_observations[year_observations["month"] == month]
        #print(month_observations)
        
        ndvi_helper = []
        for observation_path in month_observations["path"]:
            nc = xr.open_dataset(observation_path, decode_cf=False)
            ndvi_val = nc["NDVI"].values
            ndvi_helper.append(ndvi_val)
            nc.close()
        
        #print(year,month, "N/A:\n", np.argwhere(np.isnan(ndvi_helper)))
        
        month_composite = np.nanmedian(ndvi_helper, axis=0)
        #print("N/A in composit:\n", np.argwhere(np.isnan(month_composite)))
        
        composite_path = os.path.join(OUT_COMPOSITES_DIR, f"{year}_{month}.npy")
        np.save(composite_path, month_composite)
        
        out_db.loc[len(out_db)] = [year, month, composite_path]
        

out_db.to_csv(LOG_PATH)
    

#### Fill N/A in composites

In [7]:
# load composites as serie 
#composites_inventory = pd.read_csv(LOG_PATH, index_col=[1,2], header = 0)
composites_inventory = pd.read_csv(LOG_PATH, index_col=0, header = 0)
composites_inventory = composites_inventory.set_index(["year", "month"])

#print(composites_inventory)

series = []
def fillNAN(orig_series, out_series, nan_ind):
    
    offset = 1
    
    while True:
        ind_r = nan_ind+[offset,0,0] if nan_ind[0]+offset < len(orig_series) else [len(orig_series)-1, nan_ind[1], nan_ind[2]]
        right = orig_series[ind_r[0], ind_r[1], ind_r[2]]
        ind_l = nan_ind-[offset,0,0] if nan_ind[0]-offset >= 0 else [0, nan_ind[1], nan_ind[2]]
        left = orig_series[ ind_l[0], ind_l[1], ind_l[2] ]
        
        if not np.isnan(left) and not np.isnan(right):
            out_series[nan_ind[0], nan_ind[1], nan_ind[2]] = (left+right)/2
            return
        
        if np.isnan(left) and np.isnan(right):
            offset+=1
            continue
        else:
            out_series[nan_ind[0], nan_ind[1], nan_ind[2]] = left if np.isnan(right) else right
            return




for year in range(min_year,max_year+1):
    for month in range(min_month,max_month+1):
        row = composites_inventory.loc[(year,month)]
        series.append(np.load(row["path"]))

series = np.array(series)
series = series.squeeze()
print(series.shape)

# set special flags to N/A
for (value, desc) in zip (range(252,256), ["Unknown", "Snow", "Water", "Missing"]):
    flags = series == value
    print(f"{desc} ({value}): {np.sum(flags)}")
    series[flags] = np.nan 
# --------------------------

filled_series = np.copy(series)

for nan in  np.argwhere(np.isnan(series)):
    fillNAN(series, filled_series, nan)

print("NAN after fill:\n", np.argwhere(np.isnan(filled_series)))

# resave with filled
inventory_log = pd.DataFrame(columns=[ "year", "month", "path"])
index = 0
for year in range(min_year,max_year+1):
    for month in range(min_month,max_month+1):
        composite_path = os.path.join(OUT_NO_NAN_DIR, f"{year}_{month}.npy")
        np.save(composite_path, filled_series[index])
        
        inventory_log.loc[len(inventory_log)] = [year, month, composite_path]
        
        index+=1
        
inventory_log.to_csv(LOG_PATH_NO_NAN)
print("done")

(30, 100, 100)
Unknown (252): 0
Snow (253): 0
Water (254): 0
Missing (255): 2995
NAN after fill:
 []
done


#### Intra annual analysis

In [ ]:
#gfrhd